# FeatureGraph API — starter notebook

Calls the live hosted endpoint and loads the result into a pandas DataFrame.
See `featuregraph-api-usage.md` for the full field reference and what
`smooth_window` does.

In [ ]:
import os
import requests
import pandas as pd
from dotenv import load_dotenv

load_dotenv()  # reads a local .env file -- never commit that file

BASE_URL = "https://featuregraph-api.fly.dev"
API_KEY = os.environ["FEATUREGRAPH_API_KEY"]  # set in a local .env file, not here

In [ ]:
# Confirm the service is reachable before anything else -- no auth required.
r = requests.get(f"{BASE_URL}/health", timeout=10)
r.raise_for_status()
print(r.json())

In [ ]:
def run_compiler(smooth_window: int) -> dict:
    """Calls /process and returns the raw JSON response."""
    r = requests.get(
        f"{BASE_URL}/process",
        params={"smooth_window": smooth_window},
        headers={"X-API-Key": API_KEY},
        timeout=60,  # the real response is a few hundred rows of JSON; a few
                      # seconds is normal, not a sign anything is stuck
    )
    if r.status_code == 401:
        raise ValueError("401 -- check that API_KEY above is set correctly.")
    r.raise_for_status()
    return r.json()

result = run_compiler(smooth_window=100)
print(f"dataset={result['dataset']}  smooth_window={result['smooth_window']}  rows={result['row_count']}")

In [ ]:
df = pd.DataFrame(result["summary"])
df.head()

In [ ]:
# is_complete=False marks a partial object at the start/end of the recording
# with no real boundary on one side -- drop these before any real analysis,
# the same way the rest of this project treats them.
complete = df[df["is_complete"]]
print(f"{len(df)} total objects, {len(complete)} complete")
complete[["duration", "period", "amplitude_smooth", "temporal_symmetry"]].describe()

## Seeing the effect of `smooth_window`

The window isn't a neutral setting -- the same recording at two different
windows can produce very different object counts. Try it directly:

In [ ]:
for w in (1, 25, 50, 100, 200):
    r = run_compiler(smooth_window=w)
    print(f"smooth_window={w:>4}  ->  {r['row_count']} objects")